# OpenShift AI – mallin testinotebook (OpenVINO Model Server & vLLM)

Yleinen testi OpenShift AI:n single-model serving -julkaisuille.

| Osio | Runtime | Rajapinta |
|---|---|---|
| 2 | molemmat | diagnostiikka: tunnistaa runtimen ja mallin tilan |
| 3 | OpenVINO Model Server | KServe v2 REST (`/v2/models/<nimi>/infer`) |
| 4 | vLLM | OpenAI-yhteensopiva (`/v1/chat/completions`) |
| 5 | vLLM tai OVMS | embeddings: `/v1/embeddings`, OVMS `/v3/embeddings` tai ONNX-malli `/v2/.../infer` |

Täytä **osio 1**, aja **osio 2**, ja sen jälkeen mallityyppiin sopiva osio (3, 4 tai 5).

In [ ]:
%pip install -q requests numpy pillow

## 1. Asetukset

- `ENDPOINT`: mallin *external endpoint* (RHOAI → Models → deployment → Inference endpoint), ilman polkua lopussa.
- `MODEL_NAME`: deploymentin nimi pienillä kirjaimilla (sama kuin URL:n alku), esim. `navy-ships-onnx`. Diagnostiikka näyttää oikean nimen, jos tämä on väärin.
- Token kysytään erikseen, jotta se ei tallennu notebookiin. Jos token authentication ei ole päällä, paina vain Enter.

In [ ]:
import io, json, time, getpass
import numpy as np
import requests
from PIL import Image
from IPython.display import display

ENDPOINT   = "https://<deployment>-<projekti>.<cluster-domain>"
MODEL_NAME = "<deploymentin-nimi>"

TOKEN = getpass.getpass("Token (tyhjä = ei autentikointia): ").strip()
HEADERS = {"Content-Type": "application/json"}
if TOKEN:
    HEADERS["Authorization"] = f"Bearer {TOKEN}"

ENDPOINT = ENDPOINT.rstrip("/")
TIMEOUT = 120

## 2. Diagnostiikka
Tunnistaa runtimen ja näyttää mallin tilan. Lopputulos tallentuu muuttujaan `RUNTIME` (`ovms` / `vllm` / `None`).

In [ ]:
def get(path):
    try:
        r = requests.get(ENDPOINT + path, headers=HEADERS, timeout=30)
        return r.status_code, r.text
    except Exception as e:
        return None, str(e)

RUNTIME = None
checks = ["/v2", "/v2/health/ready", "/v1/config", f"/v2/models/{MODEL_NAME}", "/v1/models", "/health", "/version"]
results = {}
for p in checks:
    code_, body = get(p)
    results[p] = (code_, body)
    print(f"{code_}  {p}\n     {body[:300].strip()}\n")

# Runtimen tunnistus
if results["/v2"][0] == 200 and "OpenVINO" in results["/v2"][1]:
    RUNTIME = "ovms"
elif results["/v1/models"][0] == 200 and '"data"' in results["/v1/models"][1]:
    RUNTIME = "vllm"

print("=" * 60)
print("Runtime:", RUNTIME or "ei tunnistettu")

if RUNTIME == "ovms":
    cfg = json.loads(results["/v1/config"][1])
    for name, info in cfg.items():
        vs = info.get("model_version_status", [])
        print(f"  malli '{name}': " + (", ".join(f"v{v['version']}={v['state']}" for v in vs) or "EI VERSIOITA"))
        if not vs:
            print("    -> mallikansiosta puuttuu versiokansio: rakenteen pitää olla <polku>/1/model.onnx")
    if MODEL_NAME not in cfg:
        print(f"  ! MODEL_NAME '{MODEL_NAME}' ei löydy – käytä jotain yllä listatuista nimistä")

elif RUNTIME == "vllm":
    served = [m["id"] for m in json.loads(results["/v1/models"][1])["data"]]
    print("  vLLM palvelee malleja:", served)
    if MODEL_NAME not in served:
        print(f"  ! MODEL_NAME '{MODEL_NAME}' ei ole listalla – vaihdetaan: {served[0]}")
        MODEL_NAME = served[0]

else:
    codes = {p: c for p, (c, _) in results.items()}
    if all(c in (401, 403) for c in codes.values() if c):
        print("  -> 401/403: token puuttuu tai on väärä")
    elif all(c == 404 for c in codes.values() if c):
        print("  -> kaikki 404: pyyntö ei päädy mallipalvelimelle – tarkista ENDPOINT (Networking → Routes)")
    elif any(c is None for c in codes.values()):
        print("  -> yhteysvirhe: tarkista ENDPOINT ja että malli on Started-tilassa")

## 3. OpenVINO Model Server (ONNX / OpenVINO IR)

### 3a. Mallin metatiedot
Näyttää syötteiden ja tulosteiden nimet, muodot ja tietotyypit. `-1` tarkoittaa dynaamista ulottuvuutta.

In [ ]:
assert RUNTIME == "ovms", "Tämä osio on OpenVINO Model Serverille"

r = requests.get(f"{ENDPOINT}/v2/models/{MODEL_NAME}", headers=HEADERS, timeout=30)
r.raise_for_status()
META = r.json()
print("Versiot:", META.get("versions"))
print("\nSyötteet:")
for i in META["inputs"]:
    print(f"  {i['name']:<25} {str(i['shape']):<22} {i['datatype']}")
print("\nTulosteet:")
for o in META["outputs"]:
    print(f"  {o['name']:<25} {str(o['shape']):<22} {o['datatype']}")

### 3b. Yleinen savutesti satunnaisdatalla
Toimii mille tahansa mallille: luo jokaiselle syötteelle satunnaisen tensorin metatietojen mukaan ja tarkistaa, että inferenssi palauttaa vastauksen.
Dynaamiset ulottuvuudet (`-1`) korvataan listan `DYNAMIC_DIMS` arvoilla järjestyksessä (esim. kuvamalli `[1, 3, 224, 224]`, tekstimalli `[1, 16]`).

In [ ]:
DYNAMIC_DIMS = [1, 3, 224, 224]   # muokkaa mallin mukaan

NP_TYPES = {"FP32": np.float32, "FP16": np.float16, "FP64": np.float64,
            "INT64": np.int64, "INT32": np.int32, "INT8": np.int8, "UINT8": np.uint8, "BOOL": np.bool_}

def v2_infer(arrays: dict) -> dict:
    payload = {"inputs": [{
        "name": n, "shape": list(a.shape), "datatype": dt,
        "data": a.flatten().tolist(),
    } for n, (a, dt) in arrays.items()]}
    t0 = time.time()
    r = requests.post(f"{ENDPOINT}/v2/models/{MODEL_NAME}/infer", headers=HEADERS, json=payload, timeout=TIMEOUT)
    dt_ms = (time.time() - t0) * 1000
    if not r.ok:
        print(r.status_code, r.text[:1000])
        r.raise_for_status()
    outs = {o["name"]: np.array(o["data"]).reshape(o["shape"]) for o in r.json()["outputs"]}
    print(f"Inferenssi OK – {dt_ms:.0f} ms")
    return outs

def dummy_inputs(meta, dyn):
    arrays = {}
    for inp in meta["inputs"]:
        it = iter(dyn)
        shape = [d if d > 0 else next(it, 1) for d in inp["shape"]]
        dt = inp["datatype"]
        npt = NP_TYPES.get(dt, np.float32)
        if np.issubdtype(npt, np.floating):
            a = np.random.randn(*shape).astype(npt)
        elif npt == np.bool_:
            a = np.ones(shape, dtype=npt)
        else:
            a = np.random.randint(1, 100, size=shape).astype(npt)   # esim. token-id:t
        arrays[inp["name"]] = (a, dt)
        print(f"  {inp['name']}: {shape} {dt}")
    return arrays

outs = v2_infer(dummy_inputs(META, DYNAMIC_DIMS))
for n, a in outs.items():
    print(f"  {n}: shape {a.shape}, min {a.min():.4f}, max {a.max():.4f}")

### 3c. Kuvaluokittelu (valinnainen)
Hugging Facen kuvaluokittelijoille (ViT, ResNet, ConvNeXt …). Jos `HF_REPO` on annettu, luokkien nimet ja esikäsittelyn asetukset (koko, mean, std) haetaan mallin `config.json`- ja `preprocessor_config.json`-tiedostoista. Muuten käytetään alla olevia oletuksia.

In [ ]:
HF_REPO = "onnx-community/navy-ships-ONNX"     # tai None
IMAGE   = "https://huggingface.co/onnx-community/navy-ships-ONNX/resolve/main/images/destroyer.jpg"   # URL tai tiedostopolku

# Oletukset, jos HF-tietoja ei ole
SIZE, MEAN, STD, LABELS = (224, 224), [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], None

def hf_json(repo, fname):
    r = requests.get(f"https://huggingface.co/{repo}/resolve/main/{fname}", timeout=30)
    return r.json() if r.ok else {}

if HF_REPO:
    cfg = hf_json(HF_REPO, "config.json")
    pre = hf_json(HF_REPO, "preprocessor_config.json")
    if "id2label" in cfg:
        LABELS = [cfg["id2label"][str(i)] for i in range(len(cfg["id2label"]))]
    s = pre.get("size", {})
    if isinstance(s, dict) and "height" in s:
        SIZE = (s["width"], s["height"])
    elif isinstance(s, dict) and "shortest_edge" in s:
        SIZE = (s["shortest_edge"], s["shortest_edge"])
    elif isinstance(s, int):
        SIZE = (s, s)
    MEAN = pre.get("image_mean", MEAN)
    STD  = pre.get("image_std", STD)
print("Koko:", SIZE, "mean:", MEAN, "std:", STD)
print("Luokat:", LABELS)

def load_image(src):
    if str(src).startswith("http"):
        r = requests.get(src, timeout=30); r.raise_for_status()
        return Image.open(io.BytesIO(r.content)).convert("RGB")
    return Image.open(src).convert("RGB")

def preprocess(img):
    x = np.asarray(img.resize(SIZE, Image.BILINEAR), dtype=np.float32) / 255.0
    x = (x - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)
    return x.transpose(2, 0, 1)[np.newaxis, ...].astype(np.float32)

def softmax(z):
    e = np.exp(z - z.max()); return e / e.sum()

def classify(src, top=5):
    img = load_image(src)
    w = min(400, img.width)
    display(img.resize((w, int(img.height * w / img.width))))
    in_name = META["inputs"][0]["name"]
    logits = list(v2_infer({in_name: (preprocess(img), "FP32")}).values())[0][0]
    probs = softmax(logits.astype(np.float64))
    for i in np.argsort(probs)[::-1][:top]:
        label = LABELS[i] if LABELS else f"luokka {i}"
        print(f"  {label:<25} {probs[i] * 100:6.2f} %  {'█' * int(probs[i] * 40)}")

classify(IMAGE)

## 4. vLLM (LLM-mallit)

vLLM tarjoaa OpenAI-yhteensopivan rajapinnan. Sama endpoint toimii myös OpenAI-SDK:lla, LangChainilla ja n8n:n OpenAI-nodella (base URL = `ENDPOINT + "/v1"`).

### 4a. Chat completion

In [ ]:
assert RUNTIME == "vllm", "Tämä osio on vLLM:lle"

def chat(messages, max_tokens=256, temperature=0.2, **extra):
    payload = {"model": MODEL_NAME, "messages": messages,
               "max_tokens": max_tokens, "temperature": temperature, **extra}
    t0 = time.time()
    r = requests.post(f"{ENDPOINT}/v1/chat/completions", headers=HEADERS, json=payload, timeout=TIMEOUT)
    dt = time.time() - t0
    if not r.ok:
        print(r.status_code, r.text[:1000]); r.raise_for_status()
    data = r.json()
    u = data.get("usage", {})
    print(data["choices"][0]["message"]["content"])
    print(f"\n--- {dt:.2f} s | prompt {u.get('prompt_tokens')} tok | vastaus {u.get('completion_tokens')} tok"
          + (f" | {u['completion_tokens'] / dt:.1f} tok/s" if u.get("completion_tokens") else ""))
    return data

_ = chat([
    {"role": "system", "content": "Vastaa lyhyesti suomeksi."},
    {"role": "user", "content": "Mikä on Suomen pääkaupunki ja mitä siellä kannattaa nähdä?"},
])

### 4b. Streaming
Näyttää vastauksen sitä mukaa kuin se syntyy ja mittaa viiveen ensimmäiseen tokeniin (TTFT).

In [ ]:
def chat_stream(messages, max_tokens=256, temperature=0.2):
    payload = {"model": MODEL_NAME, "messages": messages, "max_tokens": max_tokens,
               "temperature": temperature, "stream": True}
    t0, ttft, n = time.time(), None, 0
    with requests.post(f"{ENDPOINT}/v1/chat/completions", headers=HEADERS, json=payload,
                       stream=True, timeout=TIMEOUT) as r:
        if not r.ok:
            print(r.status_code, r.text[:1000]); r.raise_for_status()
        for line in r.iter_lines(decode_unicode=True):
            if not line or not line.startswith("data: "):
                continue
            chunk = line[6:]
            if chunk.strip() == "[DONE]":
                break
            delta = json.loads(chunk)["choices"][0].get("delta", {}).get("content")
            if delta:
                if ttft is None:
                    ttft = time.time() - t0
                n += 1
                print(delta, end="", flush=True)
    total = time.time() - t0
    print(f"\n\n--- TTFT {ttft:.2f} s | {n} chunkia | yhteensä {total:.2f} s")

chat_stream([{"role": "user", "content": "Kirjoita kolme lyhyttä vinkkiä Kubernetes-podien vianetsintään."}])

### 4c. Completions-rajapinta (base-mallit ilman chat-templatea)

In [ ]:
r = requests.post(f"{ENDPOINT}/v1/completions", headers=HEADERS, timeout=TIMEOUT, json={
    "model": MODEL_NAME, "prompt": "The three primary colors are", "max_tokens": 40, "temperature": 0,
})
print(r.status_code)
print(r.json()["choices"][0]["text"] if r.ok else r.text[:1000])

### 4d. Pieni kuormatesti (valinnainen)
Lähettää `N` pyyntöä rinnakkain ja raportoi läpimenon.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

N, PARALLEL = 8, 4
prompt = [{"role": "user", "content": "Selitä yhdellä kappaleella, mitä on RAG."}]

def one(_):
    t0 = time.time()
    r = requests.post(f"{ENDPOINT}/v1/chat/completions", headers=HEADERS, timeout=TIMEOUT,
                      json={"model": MODEL_NAME, "messages": prompt, "max_tokens": 150})
    r.raise_for_status()
    return time.time() - t0, r.json()["usage"]["completion_tokens"]

t0 = time.time()
with ThreadPoolExecutor(PARALLEL) as ex:
    res = list(ex.map(one, range(N)))
wall = time.time() - t0
lat = [x[0] for x in res]; toks = sum(x[1] for x in res)
print(f"{N} pyyntöä, {PARALLEL} rinnakkain: {wall:.1f} s")
print(f"latenssi keskim. {np.mean(lat):.2f} s, max {np.max(lat):.2f} s | yhteensä {toks / wall:.1f} tok/s")

## 5. Embedding-mallit

Toimii kolmella tavalla, ja `EMBED_MODE = "auto"` valitsee automaattisesti:

| Tila | Milloin | Rajapinta |
|---|---|---|
| `openai` | vLLM:llä julkaistu embedding-malli (esim. `BAAI/bge-m3`, `intfloat/multilingual-e5-large`) | `POST /v1/embeddings` |
| `ovms_v3` | OVMS, johon embeddings-pipeline on konfiguroitu | `POST /v3/embeddings` |
| `ovms_v2` | ONNX-muotoinen embedding-malli OVMS:ssä (`<polku>/1/model.onnx`) | `POST /v2/models/<nimi>/infer`, tokenisointi ja pooling tehdään tässä notebookissa |

`ovms_v2` tarvitsee mallin tokenizerin, joten anna `EMBED_HF_REPO` (alkuperäinen HF-repo).
Huom. E5-mallit odottavat tekstien eteen etuliitteitä `query: ` / `passage: `, ja ne asetetaan `QUERY_PREFIX` / `DOC_PREFIX` -muuttujilla.

### 5a. Asetukset ja embed-funktio

In [ ]:
EMBED_MODE    = "auto"          # "auto" | "openai" | "ovms_v3" | "ovms_v2"
EMBED_HF_REPO = None            # ovms_v2: esim. "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
POOLING       = "mean"          # ovms_v2: "mean" tai "cls" (bge-mallit: cls)
QUERY_PREFIX, DOC_PREFIX = "", ""   # E5: "query: ", "passage: "

def _post(path, payload):
    return requests.post(ENDPOINT + path, headers=HEADERS, json=payload, timeout=TIMEOUT)

def _detect_embed_mode():
    if RUNTIME == "vllm":
        return "openai"
    if RUNTIME == "ovms":
        r = _post("/v3/embeddings", {"model": MODEL_NAME, "input": ["test"]})
        if r.ok:
            return "ovms_v3"
        if not EMBED_HF_REPO:
            raise ValueError("OVMS ilman /v3/embeddings-tukea: aseta EMBED_HF_REPO, jotta tokenisointi onnistuu (ovms_v2)")
        return "ovms_v2"
    raise ValueError("Runtimea ei tunnistettu – aja osio 2 tai aseta EMBED_MODE käsin")

if EMBED_MODE == "auto":
    EMBED_MODE = _detect_embed_mode()
print("EMBED_MODE:", EMBED_MODE)

_tok = None
def _ovms_v2_embed(texts):
    global _tok
    if _tok is None:
        from transformers import AutoTokenizer
        _tok = AutoTokenizer.from_pretrained(EMBED_HF_REPO)
    meta = requests.get(f"{ENDPOINT}/v2/models/{MODEL_NAME}", headers=HEADERS, timeout=30).json()
    in_names = {i["name"]: i["datatype"] for i in meta["inputs"]}
    enc = _tok(texts, padding=True, truncation=True, max_length=512, return_tensors="np")
    arrays = {n: (enc[n].astype(NP_TYPES.get(dt, np.int64)), dt) for n, dt in in_names.items() if n in enc}
    missing = set(in_names) - set(arrays)
    if missing:
        raise ValueError(f"Malli odottaa syötteitä joita tokenizer ei tuota: {missing}")
    payload = {"inputs": [{"name": n, "shape": list(a.shape), "datatype": dt, "data": a.flatten().tolist()}
                          for n, (a, dt) in arrays.items()]}
    r = _post(f"/v2/models/{MODEL_NAME}/infer", payload)
    if not r.ok:
        print(r.status_code, r.text[:1000]); r.raise_for_status()
    outs = {o["name"]: np.array(o["data"], dtype=np.float32).reshape(o["shape"]) for o in r.json()["outputs"]}
    # Valmis lause-embedding, jos malli antaa sellaisen (2D), muuten pooling token-embeddingeistä (3D)
    two_d = [a for a in outs.values() if a.ndim == 2]
    if two_d:
        return two_d[0]
    hidden = outs.get("last_hidden_state", list(outs.values())[0])
    if POOLING == "cls":
        return hidden[:, 0, :]
    mask = enc["attention_mask"][..., None].astype(np.float32)
    return (hidden * mask).sum(1) / np.clip(mask.sum(1), 1e-9, None)

def embed(texts):
    """Palauttaa normalisoidut embeddingit muodossa (len(texts), dim)."""
    t0 = time.time()
    if EMBED_MODE in ("openai", "ovms_v3"):
        path = "/v1/embeddings" if EMBED_MODE == "openai" else "/v3/embeddings"
        r = _post(path, {"model": MODEL_NAME, "input": texts})
        if not r.ok:
            print(r.status_code, r.text[:1000]); r.raise_for_status()
        data = sorted(r.json()["data"], key=lambda d: d["index"])
        E = np.array([d["embedding"] for d in data], dtype=np.float32)
    else:
        E = _ovms_v2_embed(texts)
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    print(f"{len(texts)} tekstiä -> {E.shape}  ({(time.time() - t0) * 1000:.0f} ms)")
    return E

if EMBED_MODE == "ovms_v2":
    %pip install -q transformers

### 5b. Savutesti
Tarkistaa embeddingin dimension ja sen, että sama teksti tuottaa saman vektorin.

In [ ]:
E = embed(["Hello world", "Hello world", "Hyvää huomenta"])
print("dimensio:", E.shape[1])
print("sama teksti, kosini:", round(float(E[0] @ E[1]), 4), "(pitäisi olla ~1.0)")
print("eri teksti, kosini: ", round(float(E[0] @ E[2]), 4))

### 5c. Semanttinen samankaltaisuus
Toisiaan tarkoittavien lauseiden (myös eri kielillä) pitäisi saada korkeampi pistemäärä kuin aiheesta poikkeavien.

In [ ]:
sentences = [
    "Ikkunanpesu kannattaa tehdä keväällä siitepölykauden jälkeen.",   # 0
    "Windows are best cleaned in spring after the pollen season.",   # 1 = 0 englanniksi
    "Rännit pitää tyhjentää syksyllä lehtien putoamisen jälkeen.",    # 2 sama aihepiiri
    "Kubernetes-podi jäi ImagePullBackOff-tilaan.",                   # 3 eri aihe
    "The container image could not be pulled from the registry.",     # 4 ~ 3
]
E = embed(sentences)
S = E @ E.T

print("     " + "".join(f"{j:>7}" for j in range(len(sentences))))
for i, row in enumerate(S):
    print(f"{i:>3}  " + "".join(f"{v:7.3f}" for v in row))

checks = [((0, 1), (0, 3)), ((3, 4), (3, 0))]
for (a, b), (c, d) in checks:
    ok = S[a, b] > S[c, d]
    print(f"{'OK ' if ok else 'HUOM'}  sim({a},{b})={S[a,b]:.3f} > sim({c},{d})={S[c,d]:.3f}")

### 5d. Haku (retrieval) – RAG-tyylinen testi
Upotetaan pieni dokumenttijoukko ja haetaan kysymyksille parhaiten vastaavat dokumentit.

In [ ]:
docs = [
    "Näköala+ pesee ikkunat Helsingissä, Espoossa ja Vantaalla.",
    "Rännien tyhjennys tehdään imurilla ilman tikkaita.",
    "OpenVINO Model Server vaatii mallille versiokansion, esim. 1/model.onnx.",
    "vLLM tarjoaa OpenAI-yhteensopivan rajapinnan LLM-malleille.",
    "PVC, jonka access mode on ReadWriteOnce, voidaan liittää vain yhdelle nodelle kerrallaan.",
    "Joulun valosarjat asennetaan räystäslinjaan ennen ensilunta.",
]
queries = [
    "Miksi malli ei latautunut OVMS:ään?",
    "Kuka pesee ikkunoita pääkaupunkiseudulla?",
    "Can two pods on different nodes mount the same block volume?",
]

D = embed([DOC_PREFIX + d for d in docs])
Q = embed([QUERY_PREFIX + q for q in queries])

for qi, q in enumerate(queries):
    scores = D @ Q[qi]
    print(f"\nQ: {q}")
    for di in np.argsort(scores)[::-1][:3]:
        print(f"   {scores[di]:.3f}  {docs[di]}")

### 5e. Läpimeno (valinnainen)
Embeddaa `N` tekstiä erissä ja raportoi tekstiä sekunnissa.

In [ ]:
N, BATCH = 256, 32
texts = [f"Testilause numero {i}: OpenShift AI palvelee embedding-mallia." for i in range(N)]
t0 = time.time()
for i in range(0, N, BATCH):
    _ = embed(texts[i:i + BATCH])
wall = time.time() - t0
print(f"\n{N} tekstiä {wall:.1f} s:ssa -> {N / wall:.1f} tekstiä/s (batch {BATCH})")